## Réplica Protocolo de Evaluación OPE a dataset KuaiRec

En el presente documento, se adaptará el protocolo de evaluación inicialmente realizado en OBD para distintas políticas baseline en el dataset KuaiRec.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

KUAIREC_PATH = Path("Contenido/Datasets/KuaiRec/KuaiRec 2.0/data")

smallMatrix = pd.read_csv(KUAIREC_PATH / "small_matrix.csv",
                      usecols=["user_id", "video_id", "watch_ratio"])

numberOfUsers, numberOfVideos = smallMatrix["user_id"].nunique(), smallMatrix["video_id"].nunique()

print(f"{len(smallMatrix):,} interacciones | {numberOfUsers:,} usuarios | {numberOfVideos:,} videos | "
      f"densidad {len(smallMatrix) / (numberOfUsers * numberOfVideos):.3%}")


4,676,570 interacciones | 1,411 usuarios | 3,327 videos | densidad 99.620%



Según la documentación del dataset, la variable watch_ratio se define como: 

watch_ratio = tiempo que el usuario vio el video / duración del video

Para obtener una recompensa binaria, análoga al click en OBD y KuaiRand, se sigue el criterio de Gao et al. (2022, sección 3.1.3):

"we binarize each user-video pair into either positive or negative samples accordingly to the watch time. Specifically, we define a user-video pair as a positive sample if the user's cumulative watching time of this video is greater than twice the duration of this video, i.e., the user has watched this video completely at least twice."

Dado que el tiempo acumulado de visualización dividido por la duración del video corresponde a watch_ratio, la recompensa se define como:

 $reward(u,v) = \mathbb{1}[\texttt{watch\_ratio(u,v)} \geq 2]$.

Veamos la proporción de recompensas positivas en el dataset.

In [2]:
print(smallMatrix["watch_ratio"].describe(percentiles=[.25, .5, .75, .9, .99]).round(3))
print(f"% con watch_ratio >= 2: {100 * (smallMatrix['watch_ratio'] >= 2).mean():.1f}%")

count    4676570.000
mean           0.907
std            1.362
min            0.000
25%            0.468
50%            0.769
75%            1.121
90%            1.559
99%            3.585
max          571.521
Name: watch_ratio, dtype: float64


% con watch_ratio >= 2: 4.6%


### Matriz de recompensas



Se construye la matriz $R \in$ {0,1}$^{1,411 \times 3,327}$, donde $R[u,v] = reward(u,v)$ se define como anteriormente se introduce en base al tiempo de visualización. Para completar la matriz, acorde a la metodología de Gao et al. (2022), se insertaron en el feed de cada usuario los videos que no había visto, de modo que se le mostraran todos. Así, la recompensa de cualquier recomendación es observada y no requiere estimación.

Sin embargo, la matriz no está completa totalmente. En particular, faltan 17 827 pares (0,4%), esto se debe a que "some users have explicitly blocked the videos from certain authors" (Gao et al., 2022, sec. 3.1.1). Estos pares se fijan en $R[u,v] = 0$, porque el usuario rechazó explícitamente al autor. 

In [3]:
assert not smallMatrix.duplicated(["user_id", "video_id"]).any()

usuarios = np.sort(smallMatrix["user_id"].unique())
videos = np.sort(smallMatrix["video_id"].unique())
n_usuarios, n_videos = len(usuarios), len(videos)

R = np.zeros((n_usuarios, n_videos), dtype=np.int8)   # pares faltantes (bloqueados) quedan en 0
R[pd.Index(usuarios).get_indexer(smallMatrix["user_id"]),
  pd.Index(videos).get_indexer(smallMatrix["video_id"])] = (smallMatrix["watch_ratio"] >= 2).to_numpy()

print(f"R: {R.shape} | positivos {R.mean():.4f} | pares faltantes: {n_usuarios * n_videos - len(smallMatrix):,}")

R: (1411, 3327) | positivos 0.0463 | pares faltantes: 17,827


Se obtiene una tasa positiva de 4,6% en la matriz pequeña (small_matrix).

### Metodología

Se usan como contexto las columnas de user_features.csv con entre 2 y 15 niveles, contados sobre los 1 411 usuarios de la matriz pequeña. Esto deja 21 columnas.

La codificación one-hot se ajusta sobre los 7 176 usuarios del dataset, de este modo, la matriz grande (entrenamiento) y la pequeña (evaluación) comparten las mismas 85 columnas.

In [4]:
features_usuario = pd.read_csv(KUAIREC_PATH / "user_features.csv").set_index("user_id")
niveles = features_usuario.loc[usuarios].nunique()
categoricas = [c for c in niveles.index if 2 <= niveles[c] <= 15]

dummies = pd.get_dummies(features_usuario[categoricas].astype(str), drop_first=True).astype("float32")
contexto = dummies.loc[usuarios].to_numpy()
print(f"{len(categoricas)} columnas categóricas -> {dummies.shape[1]} dummies")

21 columnas categóricas -> 85 dummies


Para que $\hat{q}$ generalice entre videos, cada acción se describe mediante 44 features (action_context de obp). Las primeras 31 corresponden a las categorías del video en codificación multi-hot (item_categories.csv), dado que un video puede pertenecer a más de una categoría.


Se agrega además la duración en segundos, relevante porque la recompensa depende de watch_ratio, que se define en relación con la duración del video. 

Las 12 features restantes son estadísticas de la plataforma obtenidas de item_daily_features.csv, sumadas sobre todo el período de recolección. Estas corresponden a la tasa de reproducciones por exhibición, a cada tipo de interacción dividida por las reproducciones y al logaritmo de las exhibiciones totales.



In [5]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

categorias = pd.read_csv(KUAIREC_PATH / "item_categories.csv").set_index("video_id").loc[videos, "feat"].apply(ast.literal_eval)
multihot = MultiLabelBinarizer().fit_transform(categorias)
duracion = (pd.read_csv(KUAIREC_PATH / "small_matrix.csv", usecols=["video_id", "video_duration"])
              .groupby("video_id")["video_duration"].median().loc[videos].to_numpy() / 1000)
diario = pd.read_csv(KUAIREC_PATH / "item_daily_features.csv")
conteos = ["show_cnt", "play_cnt", "complete_play_cnt", "valid_play_cnt", "long_time_play_cnt", "short_time_play_cnt",
           "like_cnt", "comment_cnt", "share_cnt", "follow_cnt", "collect_cnt", "report_cnt"]
totales = diario.groupby("video_id")[conteos].sum().reindex(videos)
tasas = pd.DataFrame({"play_por_show": totales["play_cnt"] / totales["show_cnt"]}, index=videos)
for c in conteos[2:]:
    tasas[c.replace("_cnt", "_por_play")] = totales[c] / totales["play_cnt"]
tasas["log_show"] = np.log1p(totales["show_cnt"])

contexto_accion = np.c_[multihot, duracion, tasas.to_numpy()].astype("float32")
print("contexto_accion:", contexto_accion.shape, "| videos sin estadísticas diarias:", totales["show_cnt"].isna().sum())

contexto_accion: (3327, 44) | videos sin estadísticas diarias: 0


### Datos de entrenamiento (Big_matrix)

Las políticas por evaluar se entrenan con la matriz grande, que según Gao et al. (2022, sec. 3.1.2) excluye todos los pares de la matriz pequeña, "to separate the training and evaluation data". De ella se conservan solo las interacciones con los 3 327 videos de la matriz pequeña, que constituyen el espacio de acciones del experimento. Así se obtienen 10 272 525 interacciones de 5 765 usuarios, con una tasa de positivos de 7,1%.

Además, se excluyen explícitamente los usuarios de la matriz pequeña, para que las políticas no aprendan de las preferencias de los usuarios con que se evalúan. 


In [6]:
grande = pd.read_csv(KUAIREC_PATH / "big_matrix.csv", usecols=["user_id", "video_id", "watch_ratio"])
grande = grande[grande["video_id"].isin(videos) & ~grande["user_id"].isin(usuarios)].reset_index(drop=True)
grande["r"] = (grande["watch_ratio"] >= 2).astype(int)
print(f"{len(grande):,} interacciones | {grande['user_id'].nunique():,} usuarios | "
      f"{grande['video_id'].nunique():,} videos | tasa de positivos {grande['r'].mean():.4f}")

10,272,525 interacciones | 5,765 usuarios | 3,327 videos | tasa de positivos 0.0710


Se tiene que la tasa de positivos obtenida en la matriz grande (7,1%) es mayor que en la pequeña (4,6%).

Esto puede deberse a que en la big_matrix los videos llegaron a los usuarios por el recomendador de la plataforma, mientras que en small_matrix se les mostraron todos los videos a cada usuario. También puede influir que se trata de usuarios distintos (5 765 y 1 411, respectivamente).

### Expresión de Valor de una Política

Dado que la matriz pequeña contiene la recompensa de todo par (usuario, video), el valor real de cualquier política $\pi$ se puede calcular de forma exacta. Tiene la siguiente expresión:

$$V(\pi) = \frac{1}{|U|} \sum_{u \in U} \sum_{v} \pi(v \mid u) , R[u, v],$$

donde $U$ son los 1 411 usuarios de la matriz pequeña. Para cada usuario se pondera la recompensa observada de cada video por la probabilidad con que la política lo recomendaría, y luego se promedia sobre los usuarios. Cada usuario pesa lo mismo, igual que en el log simulado, donde los usuarios también se sortean de manera uniforme.

In [7]:
def valor_real(pi):
    """pi: (n_videos,) si no depende del usuario, o (n_usuarios, n_videos)."""
    return float((R @ pi).mean()) if pi.ndim == 1 else float((pi * R).sum(axis=1).mean())

### Políticas por evaluar ($\pi_e$)


Como políticas a evaluar se usan las tres baselines definidas inicialmente en el proyecto (primera entrega), estas son: Random, Most Popular e IPWLearner. Random asigna probabilidad uniforme a los 3 327 videos. Most Popular ordena los videos según su tasa de positivos en la matriz grande, análogo a CTR por ítem usado en aquella entrega.

Tanto Most Popular como IPWLearner construyen una lista top-10, siguiendo el tamaño de lista de recomendación de Gao et al. (2022, sec. 5.1.2). Luego muestran uno de esos 10 videos al azar, es decir, $\pi(v \mid u) = 1/10$ para cada video de la lista. 

De este modo ambas políticas son estocásticas, en vez de recomendar siempre el mismo video. Esto reduce la varianza de los estimadores, ya que con una política determinista solo aportarían información las filas del log en que $\pi_b$ eligió exactamente ese video.

In [8]:
K = 10

tasa_pop = grande.groupby("video_id")["r"].mean().reindex(videos).to_numpy()   # tasa de positivos, como el CTR por ítem de H1

pi_random = np.full(n_videos, 1 / n_videos)

pi_popular = np.zeros(n_videos)
pi_popular[np.argsort(-tasa_pop)[:K]] = 1 / K

print("Top-10 Most Popular (tasa en matriz grande):", np.sort(tasa_pop)[::-1][:K].round(3))

Top-10 Most Popular (tasa en matriz grande): [0.621 0.617 0.613 0.6   0.59  0.59  0.589 0.585 0.58  0.578]


IPWLearner (Saito et al., 2021) aprende una política a partir de un log de bandit que incluye las propensities de la política que lo generó, por lo que no puede entrenarse directamente sobre la matriz grande. Por ello, se simula un log de 200 000 filas sobre ella. En cada fila se elige un usuario uniformemente entre los 5 765 de la matriz grande, y luego uno de sus registros al azar. Con este procedimiento, la propensity de cada fila es conocida por construcción: corresponde a la fracción de registros del usuario que tienen ese video. Así, el entrenamiento ocurre íntegramente fuera de los usuarios de evaluación.

IPWLearner entrena un clasificador ponderando cada fila por $r / \hat{\pi}_b$, de modo que las filas con recompensa 0 tienen peso nulo y no aportan a la función de pérdida. Por ello, el entrenamiento se realiza solo con las 15 597 filas positivas, lo que reduce el costo computacional. 



### Observación:

La única diferencia con usar el log completo es que los videos sin ningún positivo en la muestra no aparecen como clases. Con todo, esos videos no tienen observaciones con peso positivo, por lo que el clasificador no tendría evidencia para recomendarlos. Las filas positivas cubren 2 817 de los 3 327 videos.

Como clasificador base se usa una regresión logística con max_iter = 1000, igual que en la primera entrega. Finalmente, para cada usuario de la matriz pequeña se toman los 10 videos con mayor probabilidad predicha, y la política asigna $\pi(v \mid u) = 1/10$ a cada uno.

In [9]:
import time
from obp.policy import IPWLearner
from sklearn.linear_model import LogisticRegression

N_ENTRENAMIENTO = 200_000
rng = np.random.default_rng(0)

grande = grande.sort_values("user_id", kind="stable").reset_index(drop=True)
u_ids, inicio_u, filas_u = np.unique(grande["user_id"].to_numpy(), return_index=True, return_counts=True)
repeticiones = grande.groupby(["user_id", "video_id"])["r"].transform("size").to_numpy()

u_muestra = rng.integers(len(u_ids), size=N_ENTRENAMIENTO)
fila = inicio_u[u_muestra] + (rng.random(N_ENTRENAMIENTO) * filas_u[u_muestra]).astype(int)

a_ent = pd.Index(videos).get_indexer(grande["video_id"].to_numpy()[fila])
r_ent = grande["r"].to_numpy()[fila]
ps_ent = repeticiones[fila] / filas_u[u_muestra]
x_ent = dummies.loc[u_ids[u_muestra]].to_numpy()

positivas = r_ent == 1

# Checkpoint: el entrenamiento tarda ~16 min; si la ejecución se corta, se reutiliza el top-10 ya calculado.
CHECKPOINT_IPWL = Path("resultados/checkpoint_kuairec_ipwlearner_top10.npz")
if CHECKPOINT_IPWL.exists():
    guardado = np.load(CHECKPOINT_IPWL)
    top = guardado["top"]
    print(f"IPWLearner cargado desde {CHECKPOINT_IPWL} (entrenado con {int(guardado['n_positivas']):,} filas positivas "
          f"de {N_ENTRENAMIENTO:,} | {int(guardado['n_clases']):,} videos distintos | {float(guardado['minutos']):.1f} min)")
else:
    aprendiz = IPWLearner(n_actions=n_videos, len_list=1,
                          base_classifier=LogisticRegression(max_iter=1000, random_state=0))
    t0 = time.time()
    aprendiz.fit(context=x_ent[positivas], action=a_ent[positivas], reward=r_ent[positivas], pscore=ps_ent[positivas])
    clasificador = aprendiz.base_classifier_list[0]
    minutos = (time.time() - t0) / 60
    print(f"Entrenado con {positivas.sum():,} filas positivas de {N_ENTRENAMIENTO:,} | "
          f"{len(clasificador.classes_):,} videos distintos | {minutos:.1f} min")
    proba = clasificador.predict_proba(contexto)
    top = clasificador.classes_[np.argsort(-proba, axis=1)[:, :K]]
    Path("resultados").mkdir(exist_ok=True)
    np.savez(CHECKPOINT_IPWL, top=top, n_positivas=positivas.sum(), n_clases=len(clasificador.classes_), minutos=minutos)

pi_ipwl = np.zeros((n_usuarios, n_videos))
pi_ipwl[np.arange(n_usuarios)[:, None], top] = 1 / K

c:\Users\ddono\Desktop\2026-2\RecSys\Proyecto\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


IPWLearner cargado desde resultados\checkpoint_kuairec_ipwlearner_top10.npz (entrenado con 15,597 filas positivas de 200,000 | 2,817 videos distintos | 16.1 min)


In [10]:
politicas_e = {"Random": pi_random, "Most Popular": pi_popular, "IPWLearner": pi_ipwl}
valores = pd.Series({nombre: valor_real(pi) for nombre, pi in politicas_e.items()}, name="valor real")
valores.round(4)

Random          0.0463
Most Popular    0.6634
IPWLearner      0.0996
Name: valor real, dtype: float64

Ambas políticas superan a Random (0,0463). Most Popular alcanza 0,6634, unas 14 veces Random, lo que es coherente con que su top-10 tenga tasas de positivos de entre 0,58 y 0,62 en la matriz grande (celda 24). IPWLearner obtiene 0,0996, aproximadamente el doble de Random.

### Política de logging ($\pi_b$)



Al igual que en KuaiRand, $\pi_b$ es sintética y se define como $\pi_b(v) \propto \exp(\alpha \cdot s_v)$, donde $s_v$ es la tasa de positivos del video en la matriz grande, es decir, la misma señal de Most Popular.

De este modo, $\pi_b$ favorece los videos populares sin excluir ninguno, y con $\alpha = 0$ coincide con Random. El parámetro $\alpha$ se calibra para que el valor de $\pi_b$ sea 1,316 veces el de Random, que corresponde a la razón de CTR entre BTS y Random en OBD men. 

In [11]:
from scipy.optimize import brentq

OBD_PATH = Path("Contenido/Datasets/open_bandit_dataset/open_bandit_dataset")
ctr_obd = {pol: pd.read_csv(OBD_PATH / pol / "men" / "men.csv", usecols=["click"])["click"].mean()
           for pol in ["bts", "random"]}
razon_bts = ctr_obd["bts"] / ctr_obd["random"]
objetivo = razon_bts * valores["Random"]


def politica_logging(alpha):
    pesos = np.exp(alpha * (tasa_pop - tasa_pop.max()))
    return pesos / pesos.sum()


alpha = brentq(lambda a: valor_real(politica_logging(a)) - objetivo, 0, 1000)
pi_b = politica_logging(alpha)
print(f"razón BTS/Random = {razon_bts:.3f} | alpha = {alpha:.2f} | valor de pi_b = {valor_real(pi_b):.4f} "
      f"(objetivo {objetivo:.4f}) | pi_b mín {pi_b.min():.2e}, máx {pi_b.max():.4f}")

razón BTS/Random = 1.316 | alpha = 2.62 | valor de pi_b = 0.0609 (objetivo 0.0609) | pi_b mín 2.46e-04, máx 0.0013


Se obtiene $\alpha = 2{,}62$, con un valor de $\pi_b$ de 0,0609 y probabilidades entre $2{,}46 \cdot 10^{-4}$ y $0{,}0013$. Como todas son positivas, los pesos de importancia quedan definidos para cualquier $\pi_e$.

### Experimento

Se aplica el mismo protocolo que en OBD y KuaiRand, con logs de n = 10 000 filas y 200 semillas.

En cada semilla se sortean usuarios de manera uniforme y videos según $\pi_b$, y la recompensa se lee directamente de $R$. Luego, $\hat{q}$ se ajusta con HistGradientBoosting, con los hiperparámetros de Saito et al. (2021) y cross-fitting K = 2. 


Las tres $\pi_e$ se evalúan sobre el mismo log con los 7 estimadores, y en Switch-DR y DRos el parámetro $\lambda$ se ajusta en cada semilla minimizando el MSE estimado. Como referencia se incluye además el estimador Ingenuo, es decir, el promedio de las recompensas del log, que ignora la diferencia entre $\pi_b$ y $\pi_e$.

In [12]:
import time
from scipy.stats import ranksums
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import KFold
from obp.ope import (OffPolicyEvaluation, DirectMethod, InverseProbabilityWeighting,
                     SelfNormalizedInverseProbabilityWeighting, DoublyRobust, SelfNormalizedDoublyRobust,
                     SwitchDoublyRobustTuning, DoublyRobustWithShrinkageTuning)

LAMBDAS = [1, 5, 10, 50, 100, 500, 1000, 5000, 10000]
TUNING = dict(lambdas=LAMBDAS, tuning_method="mse", delta=0.05)


def crear_estimadores():
    return [
        InverseProbabilityWeighting(estimator_name="IPW"),
        SelfNormalizedInverseProbabilityWeighting(estimator_name="SNIPW"),
        DirectMethod(estimator_name="DM"),
        DoublyRobust(estimator_name="DR"),
        SelfNormalizedDoublyRobust(estimator_name="SNDR"),
        SwitchDoublyRobustTuning(**TUNING, estimator_name="Switch-DR"),
        DoublyRobustWithShrinkageTuning(**TUNING, estimator_name="DRos"),
    ]

def estimar_q(contexto_log, accion, recompensa, contexto_accion, semilla, filas_por_bloque=300_000):
    """q_hat(x, a) para todas las acciones, con cross-fitting K = 2 como RegressionModel de obp
    (fitting_method="normal", features = [contexto, contexto_accion[a]]).
    Como el contexto se repite entre filas, se predice una vez por contexto distinto."""
    n, n_acciones = len(accion), len(contexto_accion)
    X = np.c_[contexto_log, contexto_accion[accion]]
    q = np.zeros((n, n_acciones, 1))
    for ent, prueba in KFold(n_splits=2, shuffle=True, random_state=semilla).split(X):
        modelo = HistGradientBoostingClassifier(learning_rate=0.01, max_iter=100, max_depth=5,
                                                min_samples_leaf=10, random_state=semilla)
        modelo.fit(X[ent], recompensa[ent])
        distintos, inv = np.unique(contexto_log[prueba], axis=0, return_inverse=True)
        pred = np.empty((len(distintos), n_acciones), dtype=np.float32)
        paso = max(1, filas_por_bloque // n_acciones)
        for i in range(0, len(distintos), paso):
            bloque = distintos[i:i + paso]
            Xb = np.hstack([np.repeat(bloque, n_acciones, axis=0), np.tile(contexto_accion, (len(bloque), 1))]).astype(np.float32, copy=False)
            pred[i:i + paso] = modelo.predict_proba(Xb)[:, 1].reshape(len(bloque), n_acciones)
        q[prueba, :, 0] = pred[inv.ravel()]
    return q


N = 10_000
N_SEMILLAS = 200
ARCHIVO = Path(f"resultados/kuairec_n{N}.csv")

import gc
del grande, diario   # ya no se usan; liberan memoria antes del experimento
gc.collect()


def iteracion_kuairec(semilla, n):
    rng = np.random.default_rng(semilla)
    u = rng.integers(n_usuarios, size=n)
    a = rng.choice(n_videos, size=n, p=pi_b)
    fb = {"n_rounds": n, "n_actions": n_videos, "action": a, "reward": R[u, a].astype(int),
          "context": contexto[u], "position": np.zeros(n, dtype=int), "pscore": pi_b[a]}
    q_hat = estimar_q(fb["context"], a, fb["reward"], contexto_accion, semilla)
    filas = []
    for nombre, pi in politicas_e.items():
        dist = (np.tile(pi, (n, 1)) if pi.ndim == 1 else pi[u])[:, :, np.newaxis]
        ope = OffPolicyEvaluation(bandit_feedback=fb, ope_estimators=crear_estimadores())
        est = ope.estimate_policy_values(action_dist=dist, estimated_rewards_by_reg_model=q_hat)
        est["Ingenuo"] = fb["reward"].mean()
        filas += [{"semilla": semilla, "politica": nombre, "estimador": k, "estimado": v} for k, v in est.items()]
        del dist, ope
    del q_hat, fb
    return filas



# Checkpoint por semilla: cada semilla se agrega al CSV apenas termina. Si la ejecución se corta,
# al volver a correr el notebook se saltan las semillas ya guardadas.
Path("resultados").mkdir(exist_ok=True)
hechas = set(pd.read_csv(ARCHIVO)["semilla"]) if ARCHIVO.exists() else set()
pendientes = [s for s in range(N_SEMILLAS) if s not in hechas]
print(f"{len(hechas)} semillas ya guardadas en {ARCHIVO} | faltan {len(pendientes)}")
inicio = time.time()
n_nuevas = 0


def guardar_semilla(filas):
    global n_nuevas
    filas.to_csv(ARCHIVO, mode="a", header=not ARCHIVO.exists(), index=False)
    n_nuevas += 1
    listas = len(hechas) + n_nuevas
    if listas % 20 == 0 or n_nuevas == 1:
        print(f"{listas}/{N_SEMILLAS} semillas | {(time.time() - inicio) / 60:.1f} min en esta ejecución")


for semilla in pendientes:
    filas = pd.DataFrame(iteracion_kuairec(semilla, N))
    filas["valor_real"] = filas["politica"].map(valores)
    filas["error2"] = (filas["estimado"] - filas["valor_real"]) ** 2
    guardar_semilla(filas)
    del filas
    gc.collect()   # libera la memoria de la semilla antes de la siguiente

resultados_kr = pd.read_csv(ARCHIVO).drop_duplicates(["semilla", "politica", "estimador"])
print(f"{resultados_kr['semilla'].nunique()} semillas en {ARCHIVO}")

119 semillas ya guardadas en resultados\kuairec_n10000.csv | faltan 81


120/200 semillas | 1.1 min en esta ejecución


140/200 semillas | 20.8 min en esta ejecución


160/200 semillas | 40.7 min en esta ejecución


180/200 semillas | 61.0 min en esta ejecución


200/200 semillas | 81.0 min en esta ejecución
200 semillas en resultados\kuairec_n10000.csv


### Resultados

In [13]:
def tabla_resultados(res, valor_real):
    """RMSE, sesgo y DE relativos al valor real (%), y Wilcoxon rank-sum contra el mejor (Tabla 6, Saito et al., 2021)."""
    g = res.groupby("estimador")["estimado"]
    tabla = pd.DataFrame({
        "RMSE rel. (%)": 100 * np.sqrt(res.groupby("estimador")["error2"].mean()) / valor_real,
        "Sesgo rel. (%)": 100 * (g.mean() - valor_real) / valor_real,
        "DE rel. (%)": 100 * g.std() / valor_real,
    }).sort_values("RMSE rel. (%)")
    err = res.pivot(index="semilla", columns="estimador", values="error2")
    mejor = tabla.index[0]
    tabla[f"p vs {mejor}"] = [np.nan if e == mejor else ranksums(err[mejor], err[e]).pvalue for e in tabla.index]
    return tabla.round(3)

res = pd.read_csv(f"resultados/kuairec_n{N}.csv")
for nombre in politicas_e:
    print(f"\n=== {nombre} (valor real {valores[nombre]:.4f}) ===")
    display(tabla_resultados(res[res["politica"] == nombre], valores[nombre]))


=== Random (valor real 0.0463) ===


,RMSE rel. (%),Sesgo rel. (%),DE rel. (%),p vs SNDR
estimador,,,,
SNDR,4.252,0.061,4.262,NaN
DR,4.256,0.062,4.266,0.985
Switch-DR,4.256,0.062,4.266,0.985
DRos,4.256,0.065,4.266,0.979
IPW,4.339,0.093,4.349,0.751
SNIPW,4.388,0.099,4.398,0.666
DM,12.772,11.962,4.488,0.000
Ingenuo,32.012,31.583,5.235,0.000



=== Most Popular (valor real 0.6634) ===


,RMSE rel. (%),Sesgo rel. (%),DE rel. (%),p vs SNIPW
estimador,,,,
SNIPW,6.549,-0.210,6.562,NaN
SNDR,6.734,-0.437,6.737,0.943
DR,7.502,-0.902,7.466,0.098
Switch-DR,7.502,-0.902,7.466,0.098
IPW,11.034,-1.428,10.969,0.000
DRos,16.754,-15.728,5.789,0.000
DM,36.429,-36.216,3.944,0.000
Ingenuo,90.822,-90.821,0.365,0.000



=== IPWLearner (valor real 0.0996) ===


,RMSE rel. (%),Sesgo rel. (%),DE rel. (%),p vs DRos
estimador,,,,
DRos,13.222,-9.959,8.719,NaN
DM,16.768,-16.464,3.185,0.0
Ingenuo,38.927,-38.851,2.433,0.0
Switch-DR,41.007,5.409,40.750,0.0
DR,41.011,5.399,40.756,0.0
SNDR,44.175,6.530,43.800,0.0
IPW,44.949,6.152,44.638,0.0
SNIPW,46.189,8.051,45.596,0.0


En Random, los seis estimadores basados en pesos obtienen un RMSE relativo de alrededor de 4,3%, sin diferencias significativas en comparativa con el mejor (SNDR), mientras que DM alcanza 12,8% por su sesgo. 

En Most Popular, el mejor es SNIPW (6,5%), sin diferencias significativas con SNDR ni DR, y tanto DRos como DM quedan entre los peores por su sesgo. 

En IPWLearner, en cambio, IPW, SNIPW, DR, SNDR y Switch-DR alcanzan errores de entre 41% y 46%, explicados casi por completo por su varianza, y el mejor es DRos (13,2%), seguido de DM (16,8%).

De este modo, la desviación estándar de los estimadores basados en pesos aumenta de Random a Most Popular y de Most Popular a IPWLearner. Solo en IPWLearner esa varianza es tan alta que conviene usar estimadores que la reducen, como DRos y DM, aunque introduzcan sesgo.



**Declaración uso de IA**

Se utilizó apoyo de IA para implementar checkpoints por semilla y la liberación de memoria tras la ejecución de cada semilla, de modo que el experimento pudiera retomarse si la ejecución se interrumpía. Además como asistente corrector de código del presente notebook.

**Referencias**

Gao, C., Li, S., Lei, W., Chen, J., Li, B., Jiang, P., He, X., Mao, J., & Chua, T.-S. (2022). KuaiRec: A Fully-observed Dataset and Insights for Evaluating Recommender Systems. *Proceedings of the 31st ACM International Conference on Information & Knowledge Management (CIKM '22)*, 540–550. https://doi.org/10.1145/3511808.3557220

Saito, Y., Aihara, S., Matsutani, M., & Narita, Y. (2021). Open Bandit Dataset and Pipeline: Towards Realistic and Reproducible Off-Policy Evaluation. *NeurIPS Datasets and Benchmarks Track*. arXiv:2008.07146.

Gao, C., Li, S., Zhang, Y., et al. (2022). KuaiRand: An Unbiased Sequential Recommendation Dataset with Randomly Exposed Videos. Proceedings of the 31st ACM International Conference on Information & Knowledge Management (CIKM '22). https://doi.org/10.1145/3511808.3557624